<a href="https://colab.research.google.com/github/nasia2612/sc-alzheimer-ml/blob/main/sc_alzheimer_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 TRAIN Dataset: scRNA-seq from Alzheimer's disease prefrontal cortex
 Shape: 20,000 cells × 16,679 columns (16,678 genes + 1 label column 'tag')

 Info
 -highly sparse (mostly zeros)
 -labels: 7 brain cell types

The original file had formatting issues (separator, decimal, quote characters) that were fixed in R before saving as .rds (R's native binary format).We load it in Python via pyreadr, which reads .rds files directly into.

---




In [1]:
%%capture stored_output

!pip install pyreadr

# connect with google drive
from google.colab import drive
drive.mount('/content/drive')

#pyreadr reads rds
import pyreadr

result = pyreadr.read_r("/content/drive/MyDrive/Colab Notebooks/sc_alzheimer/sc_alz_train_set.rds")
df = result[None]   # extract the dataframe

df.head()

In [2]:
%%capture stored_output
# EXPLORE THE DATA
# basic info about the data
print(df.info())
print(df.shape)
print(df.describe())

Key findings:

-memory 2.5 (it uses a lot of memory so we need float32 conversion and gc.collect()
-25%,50% -> 0 for most genes  → confirms extreme sparsity (dropout effect in scRNA-seq)
-lets see the distribution of some genes

In [3]:
%%capture stored_output
#plot the distribution of 4 genes

import seaborn as sns
import matplotlib.pyplot as plt

#i will pick these genes
genes_to_plot = ['HES4', 'MT.ND4', 'MT.CYB', 'AGRN']

#4x4 subplots
fig,axes=plt.subplots(2,2,figsize=(12,8))
axes=axes.flatten()

for i,gene in enumerate(genes_to_plot):
    sns.histplot(df[gene], bins=50, ax=axes[i], color='steelblue', log_scale=(False, True))

    axes[i].axvline(df[gene].mean(), color='red', linestyle='--', label=f'mean={df[gene].mean():.2f}')
    axes[i].axvline(df[gene].median(), color='orange', linestyle='--', label=f'median={df[gene].median():.2f}')

    axes[i].set_title(f'{gene} expression distribution')
    axes[i].set_xlabel('Raw counts')
    axes[i].set_ylabel('Number of cells (log scale)')
    axes[i].legend()

plt.suptitle(' Distribution in scRNA-seq raw counts', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [4]:
%%capture stored_output
# Distribution of the cell types
print(df["tag"].value_counts())

In [5]:
%%capture stored_output
# do we have class imbalance?
import matplotlib.pyplot as plt

df["tag"].value_counts().plot(kind="bar", color=["blue","orange","green","red","purple","brown","pink"])
plt.title("Cell Type Distribution")
plt.xlabel("Cell Type")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

we have class imbalance so we need to use balanced in class weight

 According to  (Mathys κ.ά., 2019) the distribution matches previous data from normal cortex.

In [6]:
%%capture stored_output
# sparsity - how many zeros because of the nature of the data (dropout effect)
sparsity = (df.drop(columns=["tag"]) == 0).sum().sum()
total_values = df.shape[0] * (df.shape[1] - 1)
f" sparsity: {sparsity/total_values:.2%}"

BIG SPARSITY

OUR APPROACH:
- log2(CPM+1) normalization compresses the dynamic range, making zeros less "dominant" relative to expressed genes
-MGECT feature selection removes uninformative genes (zero
-variance across cell types), reducing noise from dropout
- Random Forest with class_weight='balanced' handles remaining sparsity robustly via ensemble of decision trees

In [7]:
%%capture stored_output
# all the cells have labels?
print((df["tag"] == "").sum())

#FINDING THE DUPLICATE CELLS

In [8]:
%%capture stored_output
# Check for duplicate rows in your raw data
#METHOD 1
duplicates = df.duplicated().sum()
print(f"Number of duplicate cells: {duplicates}")


In [9]:
%%capture stored_output
#remove the duplicates
#METHOD 2
#removing cells with the sampe tag+total_countss (keep only first occurence)

df["total_counts"]=df.drop(columns=["tag"]).sum(axis=1)

before=len(df)

df=df[~df.duplicated(subset=["tag","total_counts"],keep="first")]
after=len(df)

df.drop(columns=["total_counts"],inplace=True)
print(f"Removed: {before - after} cells")
print(f"Remaining: {after} cells")




In [10]:
%%capture stored_output
df.head()

## STEP 1: Separate X and y, then SPLIT first

Normalization (log2 CPM+1) is per-cell so technically safe before split,
but we move it AFTER split here to be methodologically cleaner.
StandardScaler MUST always be fit only on train — this prevents real data leakage.

In [11]:
%%capture stored_output
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import train_test_split

X = df.drop(columns=["tag"])
y = df["tag"]

gene_names = X.columns.tolist()

# Convert to float32 for memory efficiency
X_array = X.values.astype(np.float32)

del df, X
gc.collect()

print(f"X shape: {X_array.shape}")
print(f"y shape: {y.shape}")

In [12]:
%%capture stored_output
# STEP 2: Stratified train/val split on RAW counts (before normalization)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_array, y, test_size=0.2, random_state=42, stratify=y
)

del X_array
gc.collect()

print(f"X_train_raw: {X_train_raw.shape}")
print(f"X_val_raw:   {X_val_raw.shape}")



In [13]:
%%capture stored_output
# STEP 3: Normalization log2(CPM+1) — applied separately to train and val
# Each cell is normalized by its OWN total counts (per-cell operation)
# No information crosses between train and val

def log2_cpm_normalize(X):
    total_counts = X.sum(axis=1, keepdims=True)
    X_cpm = X / total_counts * 1e6
    X_norm = np.log2(X_cpm + 1)
    return X_norm.astype(np.float32)

X_train_norm = log2_cpm_normalize(X_train_raw)
X_val_norm   = log2_cpm_normalize(X_val_raw)

del X_train_raw, X_val_raw
gc.collect()

print("Max after norm (train):", X_train_norm.max())
print("Max after norm (val):  ", X_val_norm.max())
print(f"Memory train MB: {X_train_norm.nbytes / 1e6:.1f}")

In [14]:
%%capture stored_output
# STEP 4: MGECT — Housekeeping gene removal
# Fit ONLY on train to prevent leakage

X_train_df = pd.DataFrame(X_train_norm, columns=gene_names)
X_val_df   = pd.DataFrame(X_val_norm,   columns=gene_names)

del X_train_norm, X_val_norm
gc.collect()

# MGECT: compute median per cell type, then variance across cell types
mgect = X_train_df.copy()
mgect["tag"] = y_train.values
gene_variances = mgect.groupby("tag").median().var(axis=0)
del mgect
gc.collect()

# Genes with 0 variance across cell types = housekeeping genes (not informative)
housekeeping = gene_variances[gene_variances == 0].index
print(f"Housekeeping genes removed: {len(housekeeping)}")

X_train = X_train_df.drop(columns=housekeeping)
X_val   = X_val_df.drop(columns=housekeeping)

del X_train_df, X_val_df
gc.collect()

print(f"Genes remaining: {X_train.shape[1]}")
gene_names_filtered = X_train.columns.tolist()

In [15]:
%%capture stored_output
# mitochondrial genes check
mt_genes = [g for g in gene_names_filtered if g.startswith("MT")]
print(f"MT genes in dataset: {len(mt_genes)}")

In [16]:
%%capture stored_output
# STEP 5: StandardScaler — fit ONLY on train
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_val_scaled   = scaler.transform(X_val)          # only transform on val

del X_train, X_val
gc.collect()

print(f"Train: {X_train_scaled.shape}")
print(f"Val:   {X_val_scaled.shape}")

In [17]:
%%capture stored_output
# STEP 6: Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

# test on the validation set
y_pred = rf_model.predict(X_val_scaled)

test_accuracy = accuracy_score(y_val, y_pred)
print(f"Test accuracy: {test_accuracy:.2%}")

print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

In [18]:
%%capture stored_output
#FEATURE IMPORTANCE
# Create a series to map gene names to their importance scores
importances = pd.Series(rf_model.feature_importances_, index=gene_names_filtered)

# Sort and look at the top 10
top_10_genes = importances.sort_values(ascending=False).head(10)
print("Top 10 Most Important Genes:")
print(top_10_genes)

In [19]:
%%capture stored_output
#VISUALISATION OF THE TOP TEN IMPORTANT GENES
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
top_10_genes.sort_values().plot(kind='barh', ax=ax, color='teal', edgecolor='black')

ax.set_xlabel('Feature Importance (Gini)')
ax.set_ylabel('Gene')
ax.set_title('Top 10 Most Important Genes – Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
%%capture stored_output
from sklearn.inspection import permutation_importance

# This takes longer but is much more robust against technical artifacts
result = permutation_importance(rf_model, X_val_scaled, y_val, n_repeats=5, random_state=42)
perm_importances = pd.Series(result.importances_mean, index=gene_names_filtered)
print(perm_importances.sort_values(ascending=False).head(10))

In [ ]:
%%capture stored_output
#tuning the hypermeters
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
param_grid={
    'n_estimators':[10,100,200,300],
    'max_depth':[None,3,5,10,20],
    'max_features':["sqrt","log2",None],
    'min_samples_leaf':[1,2,4],
    'bootstrap':[True,False]
}


rf_mode=RandomForestClassifier(random_state=42)

grid_search= RandomizedSearchCV(rf_mode,param_grid,cv=10,scoring='accuracy',n_jobs=-1,verbose=2)
grid_search.fit(X_train_scaled,y_train)

best_rf=grid_search.best_estimator_

print("best parameters:",grid_search.best_params_)
print("best score:",grid_search.best_score_)

In [ ]:
%%capture stored_output
#Final train set (validation+train)

X_full=np.vstack([X_train_scaled,X_val_scaled])
y_full=pd.concat([y_train,y_val])

final_rf=RandomForestClassifier(
    grid__search.best_params_,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
    )
final_rf.fit(X_full, y_full)
print("final model trained on full dataset with best params")

In [ ]:
%%capture stored_output
#LOAD TEST SET
!pip install pyreadr
import pyreadr
from google.colab import drive
drive.mount('/content/drive')
#Read the test set
result_test=pyreadr.read_r("/content/drive/MyDrive/Colab Notebooks/sc_alzheimer/sc_alz_test_set.rds")
df_test=result_test[None]

del result_test
gc.collect()




In [ ]:
%%capture stored_output
df_test.tail()

In [ ]:
%%capture stored_output
#Normalization
#keeping the gene names
gene_names_test = df_test.columns.tolist()

#convert it to float 32 to save memory
X_test_array=df_test.values.astype(np.float32)

del df_test
gc.collect()


X_test_norm=log2_cpm_normalize(X_test_array)

del X_test_array
gc.collect()



In [ ]:
%%capture stored_output
#Remove same housekeeping genes

X_test_df=pd.DataFrame(X_test_norm,columns=gene_names_test)

del X_test_norm
gc.collect

X_test=X_test_df.drop(columns=housekeeping)

del X_test_df
gc.collect()

print(f"Genes remaining: {X_test.shape[1]}")
gene_names_filtered = X_test.columns.tolist()




In [ ]:
%%capture stored_output
#Scaling

X_test_scaled=scaler.transform(X_test)

del X_test
gc.collect()

print(f"Test: {X_test_scaled.shape}")

In [ ]:
%%capture stored_output
#Testing the data
y_pred=final_rf.predict(X_test_scaled)

#save predictions as a csv

predictions_df=pd.DataFrame({'cell_index':range(len(y_pred)),
                             'predicted_label':y_pred})


predictions_df.to_csv('/content/drive/MyDrive/datathon/predictions.csv', index=False)

print(f"Saved {len(y_pred)} predictions")
print(predictions_df['predicted_label'].value_counts())
